# 04_hybrid_search: Reciprocal Rank Fusion on Scraped Data

This notebook implements Reciprocal Rank Fusion (RRF) to merge dense FAISS semantic search results with sparse BM25 keyword rankings.


In [1]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

load_dotenv(dotenv_path=r"d:\Study\Prep\.env")

# 1. Scrape Wikipedia
url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.content, "html.parser")
corpus = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 100][:6]

# 2. Setup retrievers
embeddings = OpenAIEmbeddings()
dense_db = FAISS.from_texts(corpus, embeddings)
dense_retriever = dense_db.as_retriever(search_kwargs={"k": 3})

sparse_retriever = BM25Retriever.from_texts(corpus)
sparse_retriever.k = 3

query = "Facebook AI Research Lewis 2020"
dense_results = dense_retriever.invoke(query)
sparse_results = sparse_retriever.invoke(query)

# 3. RRF merge
def rrf_merge(dense_list, sparse_list, k=60):
    rrf_scores = {}
    for rank, doc in enumerate(dense_list):
        doc_text = doc.page_content
        rrf_scores[doc_text] = rrf_scores.get(doc_text, 0.0) + (1.0 / (k + rank + 1))
    for rank, doc in enumerate(sparse_list):
        doc_text = doc.page_content
        rrf_scores[doc_text] = rrf_scores.get(doc_text, 0.0) + (1.0 / (k + rank + 1))
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

merged = rrf_merge(dense_results, sparse_results)
print("Consolidated Hybrid Scores (Top 2):")
for text, score in merged[:2]:
    print(f"- Score {score:.5f}: '{text[:100]}...'")


C:\Users\aryan\AppData\Local\Temp\ipykernel_30920\398772098.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Consolidated Hybrid Scores (Top 2):
- Score 0.03227: 'RAG improves LLMs by incorporating information retrieval before generating responses.[3] Unlike LLMs...'
- Score 0.03227: 'Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to ret...'


### Output Explanation
- The dense index (FAISS) and sparse index (BM25) search outputs are rank-merged using RRF.
- The document containing "Facebook AI Research" and "Lewis" is consolidated as the top match.
